In [ ]:
from cemp_software_settings import load_and_apply_settings
result = load_and_apply_settings()
print(result)


USE_OPTFREQ_DATABASE = True


In [ ]:

modules_to_check = {
    'os': None,
    'subprocess': None,
    're': None,
    'csv': None,
    'concurrent.futures': None,
    'openbabel': 'openbabel', 
    'openpyxl': 'openpyxl',
    'shutil': None,
    'time': None,
    'pandas': 'pandas',
    'glob': None,
}


for module_name, conda_package in modules_to_check.items():
    try:
        
        __import__(module_name)
        print(f"Module '{module_name}' is installed.")
    except ImportError as e:
        
        print(f"Module '{module_name}' is not installed. Attempting to install...")
        
        if not conda_package:
            conda_package = module_name
        
        try:
            subprocess.check_call([sys.executable, '-m', 'conda', 'install', conda_package, '-y'])
            print(f"Module '{module_name}' installation successful.")
        except subprocess.CalledProcessError as e:
            print(f"Failed to install module '{module_name}'.", e)




In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from openbabel import openbabel, pybel
import os
import re
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AddHs
from rdkit.Chem.rdmolfiles import MolToSmiles
from openbabel import openbabel, pybel
from rdkit.Chem import RemoveHs
import shutil

In [ ]:


def calculate_charge_and_multiplicity(smiles):
    
    mol = Chem.MolFromSmiles(smiles)
    
    
    mol = Chem.AddHs(mol)
    
    
    charge = sum(atom.GetFormalCharge() for atom in mol.GetAtoms())
    
    
    multiplicity = sum([atom.GetNumRadicalElectrons() for atom in mol.GetAtoms()]) + 1
    
    return charge, multiplicity

In [ ]:

def is_monoatomic_ion(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return mol.GetNumAtoms() == 1

In [ ]:

def generate_lowest_energy_conformer_openbabel(smiles: str, num_confs: int = 50, forcefield: str = "UFF"):
    
    
    obConversion = openbabel.OBConversion()
    obConversion.SetInFormat("smi")
    
    
    obmol = openbabel.OBMol()
    obConversion.ReadString(obmol, smiles)
    
    
    obmol.AddHydrogens()
    
    
    builder = openbabel.OBBuilder()
    builder.Build(obmol)
    
    
    cs = openbabel.OBConformerSearch()
    cs.Setup(obmol, num_confs, True)
    cs.Search()
    cs.GetConformers(obmol)
    
    
    forcefield = forcefield.upper()
    if forcefield == "MMFF94":
        ff = openbabel.OBForceField.FindForceField("mmff94")
    elif forcefield == "UFF":
        ff = openbabel.OBForceField.FindForceField("uff")
    else:
        raise ValueError("Public message removed for release.")
    
    lowest_energy = float('inf')
    lowest_conf_index = None
    nconfs = obmol.NumConformers()
    
    
    for i in range(nconfs):
        obmol.SetConformer(i)
        if not ff.Setup(obmol):
            print("Public message removed for release.")
            continue
        ff.ConjugateGradients(250, 1.0e-4)
        ff.GetCoordinates(obmol)
        energy = ff.Energy()
        if energy < lowest_energy:
            lowest_energy = energy
            lowest_conf_index = i

    
    if lowest_conf_index is None:
        return None, None, None
    
    atomic_number_to_symbol = {
        1: "H",    2: "He",   3: "Li",   4: "Be",   5: "B",    6: "C",    7: "N",    8: "O",    9: "F",    10: "Ne",
        11: "Na",  12: "Mg",  13: "Al",  14: "Si",  15: "P",   16: "S",   17: "Cl",  18: "Ar",  19: "K",   20: "Ca",
        21: "Sc",  22: "Ti",  23: "V",   24: "Cr",  25: "Mn",  26: "Fe",  27: "Co",  28: "Ni",  29: "Cu",  30: "Zn",
        31: "Ga",  32: "Ge",  33: "As",  34: "Se",  35: "Br",  36: "Kr",  37: "Rb",  38: "Sr",  39: "Y",   40: "Zr",
        41: "Nb",  42: "Mo",  43: "Tc",  44: "Ru",  45: "Rh",  46: "Pd",  47: "Ag",  48: "Cd",  49: "In",  50: "Sn",
        51: "Sb",  52: "Te",  53: "I",   54: "Xe",  55: "Cs",  56: "Ba",  57: "La",  58: "Ce",  59: "Pr",  60: "Nd",
        61: "Pm",  62: "Sm",  63: "Eu",  64: "Gd",  65: "Tb",  66: "Dy",  67: "Ho",  68: "Er",  69: "Tm",  70: "Yb",
        71: "Lu",  72: "Hf",  73: "Ta",  74: "W",   75: "Re",  76: "Os",  77: "Ir",  78: "Pt",  79: "Au",  80: "Hg",
        81: "Tl",  82: "Pb",  83: "Bi",  84: "Po",  85: "At",  86: "Rn",  87: "Fr",  88: "Ra",  89: "Ac",  90: "Th",
        91: "Pa",  92: "U",   93: "Np",  94: "Pu",  95: "Am",  96: "Cm",  97: "Bk",  98: "Cf",  99: "Es", 100: "Fm",
        101: "Md", 102: "No", 103: "Lr", 104: "Rf", 105: "Db", 106: "Sg", 107: "Bh", 108: "Hs", 109: "Mt", 110: "Ds",
        111: "Rg", 112: "Cn", 113: "Nh", 114: "Fl", 115: "Mc", 116: "Lv", 117: "Ts", 118: "Og"
    }

    
    obmol.SetConformer(lowest_conf_index)
    coordinates = []
    
    for atom in pybel.ob.OBMolAtomIter(obmol):
        atomic_num = atom.GetAtomicNum()
        symbol = atomic_number_to_symbol.get(atomic_num, str(atomic_num))  
        x = atom.GetX()
        y = atom.GetY()
        z = atom.GetZ()
        coordinates.append((symbol, x, y, z))
    
    return lowest_conf_index, lowest_energy, coordinates

In [ ]:
def create_Gaussian_inputfile(df, name, smiles, nproc=10, mem=20):

    
    lowest_conf_index, lowest_energy, coordinates_list = generate_lowest_energy_conformer_openbabel(smiles, num_confs=50, forcefield="UFF")
    
    
    if lowest_conf_index is None:
        print("Public message removed for release.")
        return
    
    
    coordinate_str = ""
    for atom_type, x, y, z in coordinates_list:
        coordinate_str += f"{atom_type} {x:.6f} {y:.6f} {z:.6f}\n"

    
    filename = name.replace(' ', '_') + '.gjf'

    
    
    current_dir = os.getcwd()
    chk_file = os.path.join(current_dir, filename.replace('.gjf', '.chk'))

    
    
    charge, multiplicity = calculate_charge_and_multiplicity(smiles)

    
    
    with open(filename, 'w') as f:
        f.write(f"%nprocshared={nproc}\n")
        f.write(f"%mem={mem}GB\n")
        f.write(f"%chk={chk_file}\n")

        
        if is_monoatomic_ion(smiles):
            f.write("# b3lyp/6-311g(d,p) \n\n")
        else:
            f.write("# opt freq b3lyp/6-311g(d,p) em=gd3bj scale=0.9682 \n\n")

        f.write(f"{filename}\n\n")
        f.write(f"{charge} {multiplicity}\n")
        f.write(coordinate_str)
        
        
        f.write("\n\n")

In [ ]:
from md_qc_database_utils import (
    collect_small_molecule_rows,
    copy_reusable_results_to_success,
    get_gaussian_optfreq_database_paths,
)


In [ ]:
def create_input_files_for_missing_rows(rows):
    
    for row in rows:
        name = row['Name']
        smiles = row['SMILES']
        create_Gaussian_inputfile(None, name, smiles)
        print(f"--------{name}-------{name}----------{name}--------{name}---------{name}-------{name}----------{name}--------{name}--------")


In [ ]:
def main():
    
    df = pd.read_excel('System.xlsx')

    
    os.makedirs('Gaussian/opt+freq/success', exist_ok=True)
    os.makedirs('Gaussian/opt+freq/failure', exist_ok=True)
    os.makedirs('Gaussian/opt+freq/imaginary_frequency', exist_ok=True)

    base_dir = 'Gaussian/opt+freq'
    success_dir = os.path.join(base_dir, 'success')

    
    small_molecule_rows = collect_small_molecule_rows(df)

    if USE_OPTFREQ_DATABASE:
        optfreq_gaussian_database_path, database_excel_path = get_gaussian_optfreq_database_paths(result)
        missing_rows, found_rows = copy_reusable_results_to_success(
            rows=small_molecule_rows,
            database_directory=optfreq_gaussian_database_path,
            database_excel_path=database_excel_path,
            destination_directory=success_dir,
        )
        print("Public message removed for release.")
        print("Public message removed for release.")
    else:
        print("Public message removed for release.")
        missing_rows = small_molecule_rows

    create_input_files_for_missing_rows(missing_rows)


In [ ]:
if __name__ == '__main__':
    main()